In [0]:
# dbutils.library.restartPython()

In [0]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
from pyspark.sql import *
from pyspark.sql.types import  *
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from pyspark.sql.session import SparkSession
import joblib
# import plotly.express as px
# import plotly.io as pio

In [0]:
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics, feature_selection
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer

In [0]:
spark= SparkSession.builder.appName("Fraud").getOrCreate()

In [0]:
test_df= spark.read.csv("/Volumes/workspace/default/credit_fraud_project/archive/fraudTest.csv", header=True)
train_df= spark.read.csv("/Volumes/workspace/default/credit_fraud_project/archive/fraudTrain.csv", header=True)

In [0]:
train_df.display()

In [0]:
train_df.withColumn('unix_time', F.from_unixtime('unix_time')).display()

In [0]:
cols_to_drop = [
    "_c0",
    "first",
    "last",
    "street",
    "trans_num",
    "unix_time"
]

train_df = train_df.drop(*cols_to_drop)
test_df = test_df.drop(*cols_to_drop)

In [0]:
train_df.groupBy('is_fraud').agg(F.count("*").alias('Total')).plot.pie('is_fraud', 'Total')

In [0]:
train_df.select(*[c for c, t in train_df.dtypes if t=="string"]).display()

In [0]:
max_date= train_df.select(
    F.max(F.to_date('trans_date_trans_time'))
    ).first()[0]

train_df= train_df.withColumns({
    'trans_date_trans_time':F.col("trans_date_trans_time").cast(TimestampType()), 
    'cc_num':F.col("cc_num").cast(LongType()), 
    'amt':F.col("amt").cast(DoubleType()), 
    'zip':F.col("zip").cast(IntegerType()), 
    'lat':F.col("lat").cast(DoubleType()),
    'long':F.col("long").cast(DoubleType()),
    'city_pop':F.col("city_pop").cast(IntegerType()), 
    'dob':F.col("dob").cast(DateType()), 
    'merch_lat':F.col("merch_lat").cast(DoubleType()),
    'merch_long':F.col("merch_long").cast(DoubleType()),
    'is_fraud':F.col("is_fraud").cast(IntegerType()),
    'hour':F.hour('trans_date_trans_time'),
    'month':F.month('trans_date_trans_time'),
    'month':F.weekofyear('trans_date_trans_time')
    })
            
train_df= train_df.withColumn(
    'current_age',
    F.round(
        F.datediff(
            F.lit(max_date)
            , F.col('dob')
        ) / 365, 0
    )
    )
train_df= train_df.drop('dob')

In [0]:
max_date= test_df.select(
    F.max(F.to_date('trans_date_trans_time'))
    ).first()[0]

test_df= test_df.withColumns({
    'trans_date_trans_time':F.col("trans_date_trans_time").cast(TimestampType()), 
    'cc_num':F.col("cc_num").cast(LongType()), 
    'amt':F.col("amt").cast(DoubleType()), 
    'zip':F.col("zip").cast(IntegerType()), 
    'lat':F.col("lat").cast(DoubleType()),
    'long':F.col("long").cast(DoubleType()),
    'city_pop':F.col("city_pop").cast(IntegerType()), 
    'dob':F.col("dob").cast(DateType()), 
    'merch_lat':F.col("merch_lat").cast(DoubleType()),
    'merch_long':F.col("merch_long").cast(DoubleType()),
    'is_fraud':F.col("is_fraud").cast(IntegerType()),
    'hour':F.hour('trans_date_trans_time'),
    'month':F.month('trans_date_trans_time'),
    'month':F.weekofyear('trans_date_trans_time')
    })
            
test_df= test_df.withColumn(
    'current_age',
    F.round(
        F.datediff(
            F.lit(max_date)
            , F.col('dob')
        ) / 365, 0
    )
    )
test_df= test_df.drop('dob')

In [0]:
train_df.select(F.col('amt').cast(DoubleType()).alias('amt')).where(F.col("amt")<200).plot.hist()

In [0]:
merchant_sum= train_df.groupBy("merchant")\
            .agg(F.sum("amt").alias("total"))
top10_sum= merchant_sum.orderBy(F.desc("total")).limit(10).agg(F.sum("total").alias("top10_total")).collect()[0]['top10_total']

other_sum= train_df.select("amt").agg(F.sum("amt").alias("other_total")).collect()[0]['other_total']
labels= [
    f"Top 10 ($ {top10_sum:.2f})", 
    f"Other ($ {other_sum:.2f})"
]

plt.figure(figsize=(10,5))
plt.pie([ top10_sum, other_sum], labels=labels , autopct='%1.2f%%')

In [0]:
## Just to check Non Null values

# train_df.filter(
#     reduce(
#         lambda x,y : x & y,
#         [F.col(c).isNull() for c in train_df.columns]
#     )
# ).display()

In [0]:
train_df.columns

In [0]:
train_df.groupBy('merchant').agg(F.sum("amt").alias('Total Transaction Value')).orderBy("Total Transaction Value", ascending=False).limit(10).plot.bar(x='merchant', y='Total Transaction Value')

In [0]:
train_df.select(*[c for c, t in train_df.dtypes if t!="string"]).describe().display()

In [0]:
train_df.groupBy("job").agg(F.count("*").alias("count")).orderBy(F.desc('count')).limit(30).plot.bar(x="job", y='count')

In [0]:
plt.figure(figsize=(12,5))
sns.heatmap(train_df.select(*[c for c, t in train_df.dtypes if t!="string" and t!='date']).toPandas().corr(), annot=True, cmap='jet_r')
plt.title('Correlation Matrix')
plt.show()

In [0]:
train_df.where(F.col('is_fraud')==1).select('amt').plot.hist(x='amt')

In [0]:
train_df.where(F.col('is_fraud')==1).select('amt').pandas_api().plot.box()

In [0]:
train_df.withColumn('amt', F.col("amt").cast(DoubleType())).display()

In [0]:
train_df.select(
    F.count_distinct("merchant").alias("merchant_count"), 
    F.count_distinct("category").alias('category_count'),
    F.count_distinct("job").alias('job_count'),
    F.count_distinct("city").alias('city_count'),
    F.count_distinct("state").alias('state_count'),
    F.count_distinct("zip").alias('zip_count')
    ).display()

train_df.where(F.col("is_fraud")==1).select(
    F.count_distinct("merchant").alias("merchant_count"), 
    F.count_distinct("category").alias('category_count'),
    F.count_distinct("job").alias('job_count'),
    F.count_distinct("city").alias('city_count'),
    F.count_distinct("state").alias('state_count'),
    F.count_distinct("zip").alias('zip_count')
    ).display()

In [0]:
train_df.where(F.col("is_fraud")==1).groupby('gender').agg(F.sum("amt").alias('fraud_value')).plot.bar(x='gender', y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn(
    'year', F.year('trans_date_trans_time').alias('year')
    ).groupby('category', 'year').agg(
        F.sum("amt").alias('fraud_value')
        ).orderBy(F.desc('fraud_value')).pandas_api().pivot(
            index='category', columns='year', values='fraud_value'
            ).plot.bar(barmode='group')

In [0]:
train_df.where(F.col("is_fraud")==1).groupby('hour').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).plot.bar(x='hour', y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn('Days of month', F.dayofmonth('trans_date_trans_time')).groupby('Days of month').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).plot.bar(x='Days of month', y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn('Days of week', F.date_format('trans_date_trans_time', 'EEEE')).groupby('Days of week').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).pandas_api().set_index('Days of week').plot.bar(y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn('month', F.month('trans_date_trans_time')).groupby('month').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).pandas_api().plot.bar(x='month', y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn('weeks of year', F.weekofyear('trans_date_trans_time')).groupby('weeks of year').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).plot.bar(x='weeks of year', y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn('year', F.year('trans_date_trans_time')).groupby('year').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).plot.bar(x='year', y='fraud_value')

In [0]:
train_df.where(F.col('is_fraud')==1).withColumn(
    'year', F.year('trans_date_trans_time')
    ).groupBy('year', 'month').agg(F.sum("amt").alias('fraud_value')).orderBy('year', 'month').pandas_api()\
        .pivot(index='month', columns='year', values='fraud_value').fillna(0).reindex([
        'January', 'February', 'March', 'April',
        'May', 'June', 'July', 'August',
        'September', 'October', 'November', 'December'
    ]).plot.bar(barmode='group')

In [0]:
(
    train_df.where(F.col('is_fraud')==1).withColumn(
        'year', F.year('trans_date_trans_time')
    ).groupBy('year', 'hour').agg(
        F.sum("amt").alias('fraud_value')
        ).orderBy('year', 'hour')
    .pandas_api().pivot(index='hour', columns='year', values='fraud_value').fillna(0)
).plot.bar(barmode='group')

In [0]:
(
    train_df.where(F.col('is_fraud')==1).withColumn(
        'year', F.year('trans_date_trans_time')
    ).groupBy('year', 'hour').agg(
        F.count('trans_date_trans_time').alias('count')
        ).orderBy('year', 'hour')
    .pandas_api().pivot(index='hour', columns='year', values='count').fillna(0)
).plot.bar(barmode='group').update_layout(
    title='Fraud Transactions Count by Hour and Year',
    xaxis_title='Hour',
    yaxis_title='Count'
)

In [0]:
# Average Fraud Amount by Hour and Year

(
    (
        train_df.where(F.col('is_fraud')==1).withColumn(
            'year': F.year('trans_date_trans_time')
        ).groupBy('year', 'hour').agg(
            F.sum("amt").alias('fraud_value')
            ).orderBy('year', 'hour')
        .pandas_api().pivot(index='hour', columns='year', values='fraud_value').fillna(0)
    ) 
    / 
    (
        train_df.where(F.col('is_fraud')==1).withColumn(
            'year': F.year('trans_date_trans_time')
        ).groupBy('year', 'hour').agg(
            F.count('trans_date_trans_time').alias('count')
            ).orderBy('year', 'hour')
        .pandas_api().pivot(index='hour', columns='year', values='count').fillna(0)
    )
).plot.line().update_layout(
    title='Average Fraud Transactions Value by Hour',
    xaxis_title='Hour',
    yaxis_title='Average Fraud Transaction'
)

In [0]:
train_df.where(F.col('is_fraud')==1).select('current_age').plot.hist(bins=5).update_layout(
    title='Fraud transaction distribution by age',
    xaxis_title= 'Age Distribution',
    yaxis_title= 'Fraud Transactions',
    width=1000
)